<p align="center">
  <img src="https://www.centralbank.ie/images/default-source/news-articles/press-releases/2026/innovation-data-challenge.png?sfvrsn=a2436d1a_4" width="700"/>
</p>

## 👋 Welcome to the Innovation Data Challenge 2026

The Central Bank of Ireland and Banca d’Italia have launched their first joint Innovation Data Challenge to explore how data can drive innovation in payments and financial systems.

In this hackathon, you will work with **synthetic datasets and APIs** to explore, analyse, and prototype new ideas.

---

🎯 Your Mission

Use the available datasets and/or APIs to:

- Explore meaningful patterns in the data  
- Identify insights relevant to payments, innovation, or financial systems  
- Build a prototype, analysis, dashboard, or model  
- Clearly explain your thinking and approach  

There is no single “correct” answer — creativity and clarity matter.

---

💡 What Makes a Strong Submission?

Since this is an innovation challenge, strong projects typically:

- Start with a clear question or hypothesis  
- Use data thoughtfully (not just lots of it)  
- Explain assumptions and limitations  
- Present findings clearly (charts, summaries, demo)  
- Propose next steps or future improvements  

---

🛠 What This Notebook Will Help You Do

1. Connect to the database  
2. Explore featured datasets  
3. Preview and analyse tables  
4. (Optional) Connect via API  
5. Build your own solution  

---

> 🚀 Tip: Run each cell from top to bottom.  
> You’re encouraged to modify, experiment, and extend this notebook.


## 1️⃣ Connecting to the Database 🗄️

Before exploring datasets, we first need to connect to the PostgreSQL database that contains the synthetic challenge data.

This environment is already configured with secure credentials using environment variables:

- `DB_USER`
- `DB_HOST`
- `DB_PASSWORD`
- `DB_DATABASE`

You do **not** need to manually enter credentials.

---

🔎 What This Section Does

- Establishes a connection to the database
- Creates a cursor to run SQL queries
- Confirms that everything is working

If this step works, the rest of the notebook will run smoothly.


In [2]:
# Connect to Postgres

import os
import psycopg2

def newCursor():
    try:
        connection = psycopg2.connect(
            user=os.environ["DB_USER"],
            host=os.environ["DB_HOST"],
            password=os.environ["DB_PASSWORD"],
            port="5432",
            database=os.environ["DB_DATABASE"]
        )

        cursor = connection.cursor()
        return cursor

    except (Exception, psycopg2.Error) as error:
        print("Error while connecting to PostgreSQL", error)
        return error


In [3]:
# Test database connection

cursor = newCursor()

if isinstance(cursor, psycopg2.extensions.cursor):
    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()
    print("✅ Successfully connected to PostgreSQL")
    print("Database version:", db_version[0])
else:
    print("❌ Connection failed. Please check environment configuration.")


✅ Successfully connected to PostgreSQL
Database version: PostgreSQL 16.8 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 7.3.1 20180712 (Red Hat 7.3.1-17), 64-bit


## 2️⃣ Explore Available Tables 🔎

Now that you're connected to the database, let’s explore what data is available.

We recommend starting with the **featured challenge tables** listed below.

You can also list all tables in the database if you'd like to explore further.


In [14]:
# Featured Challenge Tables

FEATURED_TABLES = [
    "D01_transactions_2024-01",
    "D01_transactions_2024-02",
    "D01_transactions_2024-03",
    "D01_transactions_2024-04",
    "D01_transactions_2024-05",
    "D01_transactions_2024-06",
    "D01_transactions_2024-07",
    "D01_transactions_2024-08",
    "D01_transactions_2024-09",
    "D01_transactions_2024-10",
    "D01_transactions_2024-11",
    "D01_transactions_2024-12",
    "D02_payment_instruments",
    "D03_settlement_logs",
    "D04_fraud_events",
    "D05_system_outages",
    "D06_psps",
    "D07_access_channels",
    "D08_cross_border_sepa",
    "D09_macro_aggregates",
    "D10_profiles",
]

print("⭐ Featured Tables loaded:", len(FEATURED_TABLES))


⭐ Featured Tables loaded: 21


In [18]:
# Fetch tables and validate featured tables

from collections import defaultdict

c = newCursor()

q = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
"""
c.execute(q)
records = c.fetchall()

user_records = [
    (schema, table) for schema, table in records
    if schema not in ("pg_catalog", "information_schema")
]

schemas_by_table = defaultdict(list)
for schema, table in user_records:
    schemas_by_table[table].append(schema)

TABLE_LOOKUP = {
    table: f"{schemas[0]}.{table}"
    for table, schemas in schemas_by_table.items()
}

print("🔎 Checking Featured Tables...\n")

found = 0
for table in FEATURED_TABLES:
    if table in TABLE_LOOKUP:
        print(f"✅ {table}")
        found += 1
    else:
        print(f"❌ {table} (not found)")

print(f"\nMatched {found} out of {len(FEATURED_TABLES)} featured tables.")


🔎 Checking Featured Tables...

✅ D01_transactions_2024-01
✅ D01_transactions_2024-02
✅ D01_transactions_2024-03
✅ D01_transactions_2024-04
✅ D01_transactions_2024-05
✅ D01_transactions_2024-06
✅ D01_transactions_2024-07
✅ D01_transactions_2024-08
✅ D01_transactions_2024-09
✅ D01_transactions_2024-10
✅ D01_transactions_2024-11
✅ D01_transactions_2024-12
✅ D02_payment_instruments
✅ D03_settlement_logs
✅ D04_fraud_events
✅ D05_system_outages
✅ D06_psps
✅ D07_access_channels
✅ D08_cross_border_sepa
✅ D09_macro_aggregates
✅ D10_profiles

Matched 21 out of 21 featured tables.


In [16]:
# List all tables

c = newCursor()

q = """
SELECT table_name
FROM information_schema.tables
"""
c.execute(q)
records = c.fetchall()

# Filter out PostgreSQL system tables
user_tables = [
    r[0] for r in records
    if not r[0].startswith("pg_")
]

if not user_tables:
    print("❌ No user tables found.")
else:
    print(f"📚 Total user tables found: {len(user_tables)}\n")
    for table in user_tables:
        print("-", table)


📚 Total user tables found: 693

- q_airport_distance
- stocks_yahoo_finance
- q_ApprovalRequest
- default
- accounts
- card2
- cards
- q_airportdistance
- channel
- audit_log_schema
- clients
- q_approvalrequest
- collations
- information_schema_catalog_name
- applicable_roles
- domain_constraints
- administrable_role_authorizations
- collation_character_set_applicability
- attributes
- character_sets
- column_udt_usage
- check_constraint_routine_usage
- column_column_usage
- check_constraints
- constraint_table_usage
- column_domain_usage
- columns
- column_privileges
- constraint_column_usage
- domain_udt_usage
- domains
- enabled_roles
- key_column_usage
- parameters
- table_privileges
- referential_constraints
- sql_features
- role_column_grants
- routine_column_usage
- routine_privileges
- role_routine_grants
- routine_routine_usage
- routine_sequence_usage
- routine_table_usage
- routines
- schemata
- sequences
- role_table_grants
- table_constraints
- triggers
- tables
- trigger

## 3️⃣ Preview a Table 📄

Now that you can see the available tables, let’s preview rows from a table you choose.

This section will help you:
- Pick a table (recommended: start with a featured one)
- See the first few rows (`LIMIT`)
- Inspect column names
- Optionally load results into a pandas DataFrame (if installed)

> Tip: Start with a small preview (10–50 rows). You can always pull more later.


In [21]:
# Preview rows from a selected table

from psycopg2 import sql

# ✅ Choose a table
TABLE = FEATURED_TABLES[2]  # change index e.g. FEATURED_TABLES[3]
LIMIT = 10

if TABLE not in TABLE_LOOKUP:
    raise ValueError(f'❌ Table "{TABLE}" not found. Run Section 2 cells first.')

# Get schema + table name internally
schema, table_name = TABLE_LOOKUP[TABLE].split(".", 1)

c = newCursor()

# Safe query (handles schema + hyphenated table names correctly)
query = sql.SQL("SELECT * FROM {}.{} LIMIT %s").format(
    sql.Identifier(schema),
    sql.Identifier(table_name)
)

c.execute(query, (LIMIT,))
rows = c.fetchall()

# Column names
colnames = [desc[0] for desc in c.description]

print(f"📄 Table: {TABLE}")
print(f"Columns ({len(colnames)}): {colnames}\n")
print(f"Showing {min(LIMIT, len(rows))} rows:\n")

# Pretty display if pandas is available
try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=colnames)
    display(df)
except ImportError:
    for r in rows[:5]:
        print(r)
    if len(rows) > 5:
        print(f"\n... ({len(rows)-5} more rows)")


📄 Table: D01_transactions_2024-03
Columns (16): ['transaction_id', 'timestamp', 'payer_id', 'payee_id', 'amount', 'currency', 'sender_psp_id', 'receiver_psp_id', 'payment_instrument', 'channel', 'corridor', 'counterparty_type', 'merchant_category', 'status', 'settlement_date', 'is_instant']

Showing 10 rows:



,transaction_id,timestamp,payer_id,payee_id,amount,currency,sender_psp_id,receiver_psp_id,payment_instrument,channel,corridor,counterparty_type,merchant_category,status,settlement_date,is_instant
0,59949ce8-052b-441c-8d06-ef372f35b155,2024-03-01 00:00:50,IT_CON_0741439,IT_MER_0979508,20.14,EUR,IT_EMI__0213,IT_EMI__0161,Card,Contactless,Domestic_IT,P2B,5533.0,Completed,2024-03-04,False
1,86b9d4f6-c739-4aa2-bb51-c8a999a82531,2024-03-01 00:01:09,IT_CON_0669583,IT_MER_0972005,34.96,EUR,IT_EMI__0163,IT_COMM_0058,Direct_Debit,Web_Portal,Cross_Border_IT_IE,P2B,7298.0,Completed,2024-03-04,False
2,a40f1b2d-27d7-4951-92b0-5673b3a0416e,2024-03-01 00:01:42,IE_CON_0236198,IE_MER_0494762,270.58,EUR,IE_CRED_0088,IE_PI_0226,E_Money,Web_Portal,Domestic_IE,P2B,7011.0,Completed,2024-03-01,True
3,fa1f24c5-9e9d-41e0-acd4-a102a9972ac4,2024-03-01 00:02:09,IT_CON_0561568,IT_MER_0956109,17.44,EUR,IT_CRED_0116,IT_EMI__0185,Card,Contactless,Domestic_IT,P2B,5977.0,Completed,2024-03-04,False
4,b26cd45b-3d3a-41a5-8c65-17eb7287dccf,2024-03-01 00:02:17,IT_CON_0548527,IT_MER_0950976,40.34,EUR,IT_CRED_0140,IT_CRED_0153,Card,Web_Portal,Domestic_IT,P2B,5499.0,Completed,2024-03-04,False
5,c4e624b6-ad11-430e-9adb-af941e2fdcf4,2024-03-01 00:03:01,IE_CON_0434344,IT_MER_0965171,22.91,EUR,IE_EMI__0141,IT_CRED_0147,Card,Contactless,Cross_Border_IE_IT,P2B,5533.0,Completed,2024-03-04,False
6,8b48c967-77d8-43fc-b80c-e43e85590bdc,2024-03-01 00:03:35,IT_CON_0743337,IT_MER_0990882,12.27,EUR,IT_COMM_0046,IT_PI_0362,Direct_Debit,Web_Portal,Domestic_IT,P2B,6011.0,Completed,2024-03-04,False
7,94861e27-101b-4de5-b8a7-800d6ab5480e,2024-03-01 00:03:41,IE_CON_0369194,IE_MER_0495562,36.66,EUR,IE_COMM_0030,IE_EMI__0151,Direct_Debit,Web_Portal,Domestic_IE,P2B,5942.0,Completed,2024-03-04,False
8,51137ee5-7879-4e84-9c59-36fb3a75cadc,2024-03-01 00:03:42,IT_CON_0570082,IT_MER_0952179,33.32,EUR,IT_EMI__0195,IT_CRED_0148,Credit_Transfer,Web_Portal,Domestic_IT,P2B,5045.0,Completed,2024-03-04,False
9,538096be-4584-437d-9118-74904f99cb15,2024-03-01 00:04:11,IE_CON_0377946,IE_MER_0457875,49.91,EUR,IE_EMI__0158,IE_COMM_0056,Card,Contactless,Domestic_IE,P2B,4816.0,Completed,2024-03-04,False


## 4️⃣ Inspect Table Structure 🧱

Before writing more complex queries, it’s helpful to understand the structure of your selected table.

This section shows:

- Column names
- Data types
- Whether values can be NULL

Understanding your schema will help you:
- Choose grouping variables
- Filter correctly
- Identify join keys
- Detect timestamps or numeric measures


In [22]:
# Inspect column structure for selected table

schema, table_name = TABLE_LOOKUP[TABLE].split(".", 1)

c = newCursor()

q = """
SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = %s
  AND table_name = %s
ORDER BY ordinal_position;
"""

c.execute(q, (schema, table_name))
columns = c.fetchall()

print(f"🧱 Structure for table: {TABLE}\n")

for col_name, data_type, nullable in columns:
    print(f"- {col_name} | {data_type} | nullable: {nullable}")


🧱 Structure for table: D01_transactions_2024-03

- transaction_id | character varying | nullable: YES
- timestamp | character varying | nullable: YES
- payer_id | character varying | nullable: YES
- payee_id | character varying | nullable: YES
- amount | numeric | nullable: YES
- currency | character varying | nullable: YES
- sender_psp_id | character varying | nullable: YES
- receiver_psp_id | character varying | nullable: YES
- payment_instrument | character varying | nullable: YES
- channel | character varying | nullable: YES
- corridor | character varying | nullable: YES
- counterparty_type | character varying | nullable: YES
- merchant_category | numeric | nullable: YES
- status | character varying | nullable: YES
- settlement_date | character varying | nullable: YES
- is_instant | boolean | nullable: YES


## 5️⃣ Build, Explore, and Innovate 🚀

You’re now fully set up.

You can:

- Query any featured table
- Join tables together
- Aggregate and group data
- Build charts and dashboards
- Create models
- Develop a prototype
- Generate insights and policy ideas

---

💡 Suggestions for Next Steps

Here are some directions you might explore:

- Analyse transaction volumes across months
- Investigate fraud patterns
- Examine system outages and impact
- Explore cross-border SEPA activity
- Compare macro aggregates with transactional data
- Identify PSP behaviour patterns
- Segment profiles and usage trends

---

🧰 You Are Free To Install Packages

You may install any packages needed for your analysis, for example:

```python
%pip install pandas matplotlib seaborn plotly scikit-learn
